# ReferSplat: Referring Segmentation in 3D Gaussian Splatting
**ICML 2025 Oral**

이 노트북은 ReferSplat을 Google Colab 환경에서 실행하기 위한 전체 파이프라인입니다.

**파이프라인:**
1. 환경 설정 및 의존성 설치
2. 데이터셋 준비 (LERF-OVS)
3. 사전학습 3DGS 체크포인트 준비
4. ReferSplat 학습 (Stage 1)
5. 렌더링 (Inference)
6. 평가 (mIoU)

## 0. GPU 확인

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. 환경 설정

### 1-1. 경로 설정
**여기서 데이터 경로를 수정하세요.**

In [ ]:
import os

# ============================================================
# [수정 필요] 데이터 및 출력 경로 설정
# ============================================================
DATA_ROOT = "/content/data/lerf-ovs/Ref-lerf"  # LERF-OVS 데이터셋 루트 경로
OUTPUT_ROOT = "/content/output"                # 학습 결과 저장 경로 (Colab 로컬)
CHECKPOINT_ROOT = "/content/checkpoints"       # 사전학습 3DGS 체크포인트 경로
REPO_DIR = "/content/ReferSplat"               # 레포 클론 경로
GITHUB_REPO = "BAEJUNHAK/ReferSplat"          # GitHub 레포 (username/repo)

# Google Drive 저장 경로 (세션 끊겨도 유지)
DRIVE_SAVE_DIR = "/content/drive/MyDrive/ReferSplat_checkpoints"

# 사용할 장면 선택 (figurines, ramen, teatime, waldo_kitchen)
SCENE = "ramen"

# 저자 제공 완성 모델 사용 여부
USE_AUTHOR_MODEL = True  # False로 바꾸면 직접 학습
AUTHOR_CKPT_MAP = {
    "ramen": "ramen.pth",
    "waldo_kitchen": "waldo_kitchen.pth",
}

# 평가/분석 대상 장면 (gt_mask가 있는 장면만)
ALL_SCENES = ["ramen"]
# ALL_SCENES = ["ramen", "waldo_kitchen"]  # waldo_kitchen 추가 시 주석 해제

# 사전학습 체크포인트 파일명 매핑 (HuggingFace 다운로드 구조에 맞춤)
CKPT_NAME_MAP = {
    "figurines": "figurineschkpnt30000.pth",
    "ramen": "ramenchkpnt30000.pth",
    "teatime": "teatimechkpnt30000.pth",
    "waldo_kitchen": "kitchenchkpnt30000.pth",
}

# 파생 경로 (자동 설정)
SCENE_DATA_PATH = os.path.join(DATA_ROOT, SCENE)
SCENE_OUTPUT_PATH = os.path.join(OUTPUT_ROOT, SCENE)
PRETRAINED_CKPT = os.path.join(CHECKPOINT_ROOT, CKPT_NAME_MAP[SCENE])

os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(CHECKPOINT_ROOT, exist_ok=True)

print(f"Scene: {SCENE}")
print(f"Data path: {SCENE_DATA_PATH}")
print(f"Output path: {SCENE_OUTPUT_PATH}")
print(f"Pretrained checkpoint: {PRETRAINED_CKPT}")
print(f"Drive save dir: {DRIVE_SAVE_DIR}")
print(f"GitHub repo: {GITHUB_REPO}")

# 평가 출력 경로 (저자 모델 사용 시 자동 변경됨)
if USE_AUTHOR_MODEL:
    EVAL_OUTPUT_PATH = os.path.join(OUTPUT_ROOT, SCENE + "_author")
else:
    EVAL_OUTPUT_PATH = SCENE_OUTPUT_PATH

### 1-2. Google Drive 마운트 및 체크포인트 복원
이전 세션에서 학습한 체크포인트를 Drive에서 자동으로 불러옵니다.

In [ ]:
import glob, shutil
from google.colab import drive

# Google Drive 마운트
drive.mount('/content/drive')
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

if USE_AUTHOR_MODEL:
    print("저자 모델 사용 모드 - Drive 체크포인트 복원 불필요")
else:
    # Drive에 이전 세션의 체크포인트가 있으면 로컬로 복원
    drive_scene_dir = os.path.join(DRIVE_SAVE_DIR, SCENE)
    if os.path.exists(drive_scene_dir):
        drive_ckpts = sorted(glob.glob(os.path.join(drive_scene_dir, "chkpnt*.pth")))
        if drive_ckpts:
            os.makedirs(SCENE_OUTPUT_PATH, exist_ok=True)
            for ckpt in drive_ckpts:
                dst = os.path.join(SCENE_OUTPUT_PATH, os.path.basename(ckpt))
                if not os.path.exists(dst):
                    print(f"Drive에서 복원: {os.path.basename(ckpt)}")
                    shutil.copy2(ckpt, dst)
            print(f"복원 완료! 학습을 건너뛰고 바로 추론 가능합니다.")
        else:
            print("Drive에 저장된 체크포인트가 없습니다. 학습이 필요합니다.")
    else:
        print("Drive에 저장된 체크포인트가 없습니다. 학습이 필요합니다.")

### 1-3. 레포지토리 클론 및 CUDA 서브모듈 빌드

In [ ]:
# 레포지토리 클론 (자신의 GitHub 레포에서)
%cd /content
if not os.path.exists(REPO_DIR):
    !git clone --recursive https://github.com/{GITHUB_REPO}.git
else:
    # 이미 클론된 경우 최신 변경사항 pull
    print("Repository already exists, pulling latest changes...")
    !cd {REPO_DIR} && git pull && git submodule update --init --recursive
%cd {REPO_DIR}
!git log --oneline -3

In [ ]:
# 필수 패키지 설치
!pip install plyfile==0.8.1 jaxtyping open-clip-torch tensorboard -q

In [ ]:
# transformers 설치 (BERT 사용)
# 원본은 transformers==3.2.0이지만 Colab Python 3.10+와 호환 안됨
# BertTokenizer/BertModel API는 호환되므로 최신 버전 사용
!pip install transformers -q

In [ ]:
# CUDA 서브모듈 빌드: langsplat-rasterization
print("=" * 50)
print("Building langsplat-rasterization (CUDA)...")
print("=" * 50)
%cd {REPO_DIR}/submodules/langsplat-rasterization
!pip install --no-build-isolation .
%cd {REPO_DIR}

In [ ]:
# CUDA 서브모듈 빌드: simple-knn
print("=" * 50)
print("Building simple-knn (CUDA)...")
print("=" * 50)
%cd {REPO_DIR}/submodules/simple-knn
!pip install --no-build-isolation .
%cd {REPO_DIR}

In [ ]:
# segment-anything-langsplat 빌드
print("=" * 50)
print("Building segment-anything-langsplat...")
print("=" * 50)
%cd {REPO_DIR}/submodules/segment-anything-langsplat
!pip install --no-build-isolation .
%cd {REPO_DIR}

In [ ]:
# 빌드 검증
import sys
sys.path.insert(0, REPO_DIR)

try:
    import diff_gaussian_rasterization
    print("[OK] diff_gaussian_rasterization")
except ImportError as e:
    print(f"[FAIL] diff_gaussian_rasterization: {e}")

try:
    from simple_knn._C import distCUDA2
    print("[OK] simple_knn")
except ImportError as e:
    print(f"[FAIL] simple_knn: {e}")

try:
    from transformers import BertTokenizer, BertModel
    print("[OK] transformers (BERT)")
except ImportError as e:
    print(f"[FAIL] transformers: {e}")

## 2. 데이터셋 다운로드

LERF-OVS 데이터셋 (4개 장면: figurines, ramen, teatime, waldo_kitchen)

**다운로드 옵션:**
- HuggingFace: https://huggingface.co/datasets/FudanCVL/Ref-Lerf
- Baidu: https://pan.baidu.com/s/1D9yDNfUrK-d8eGO33Avkpg?pwd=ugs3

**데이터 구조 (각 장면별):**
```
<scene>/
├── images/              # RGB 이미지
├── sparse/0/            # COLMAP (cameras.bin, images.bin, points3D.bin)
├── json/
│   ├── train_json/      # frame_XXXXX.json (학습 어노테이션)
│   └── test_json/       # frame_XXXXX.json (테스트 어노테이션)
└── gt_mask/             # GT 세그멘테이션 마스크 (PNG)
```

In [ ]:
# ============================================================
# 데이터 다운로드 (HuggingFace) + 압축 해제 + 구조 정리
# ============================================================
import os

HF_DOWNLOAD_DIR = "/content/data/lerf-ovs"

# Step 1: HuggingFace에서 다운로드
if not os.path.exists(os.path.join(HF_DOWNLOAD_DIR, "Ref-lerf.zip")):
    !pip install huggingface_hub -q
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id="FudanCVL/Ref-Lerf",
        repo_type="dataset",
        local_dir=HF_DOWNLOAD_DIR,
    )
    print("다운로드 완료!")
else:
    print("이미 다운로드됨. 스킵.")

# zip 파일 위치 확인 (HF 구조에 따라 다를 수 있음)
import glob
zip_base = HF_DOWNLOAD_DIR
ref_zip = glob.glob(os.path.join(HF_DOWNLOAD_DIR, "**/Ref-lerf.zip"), recursive=True)
if ref_zip:
    zip_base = os.path.dirname(ref_zip[0])
print(f"ZIP 파일 위치: {zip_base}")

# Step 2: Ref-lerf.zip 압축 해제
DATASET_DIR = os.path.join(zip_base, "Ref-lerf")
if not os.path.exists(os.path.join(DATASET_DIR, "ramen")):
    print("Ref-lerf.zip 압축 해제 중...")
    !cd {zip_base} && unzip -q -o Ref-lerf.zip
    print("완료!")
else:
    print("Ref-lerf.zip 이미 해제됨.")

# DATA_ROOT 업데이트
DATA_ROOT = DATASET_DIR
SCENE_DATA_PATH = os.path.join(DATA_ROOT, SCENE)
print(f"DATA_ROOT = {DATA_ROOT}")

# Step 3: 각 scene에 sparse/ 심볼릭 링크 생성 (distorted/sparse/ → sparse/)
# 코드가 <scene>/sparse/0/ 경로를 기대하기 때문
for scene_name in ["figurines", "ramen", "teatime", "waldo_kitchen"]:
    scene_dir = os.path.join(DATASET_DIR, scene_name)
    if not os.path.isdir(scene_dir):
        continue
    sparse_link = os.path.join(scene_dir, "sparse")
    distorted_sparse = os.path.join(scene_dir, "distorted", "sparse")
    if not os.path.exists(sparse_link) and os.path.exists(distorted_sparse):
        os.symlink(distorted_sparse, sparse_link)
        print(f"[{scene_name}] sparse/ -> distorted/sparse/ 링크 생성")
    elif os.path.exists(sparse_link):
        print(f"[{scene_name}] sparse/ 이미 존재")

# Step 4: ramen_masks.zip 해제 → ramen/gt_mask/
ramen_gt = os.path.join(DATASET_DIR, "ramen", "gt_mask")
ramen_zip = os.path.join(zip_base, "ramen_masks.zip")
if not os.path.exists(ramen_gt) and os.path.exists(ramen_zip):
    print("ramen_masks.zip 압축 해제 중...")
    !unzip -q -o {ramen_zip} -d /tmp/_tmp_ramen
    !mv /tmp/_tmp_ramen/gt_mask {ramen_gt}
    !rm -rf /tmp/_tmp_ramen
    print("ramen gt_mask 완료!")
elif os.path.exists(ramen_gt):
    print("ramen gt_mask 이미 존재.")

# Step 5: waldo_kitchen_mask.zip 해제 → waldo_kitchen/gt_mask/
waldo_gt = os.path.join(DATASET_DIR, "waldo_kitchen", "gt_mask")
waldo_zip = os.path.join(zip_base, "waldo_kitchen_mask.zip")
if not os.path.exists(waldo_gt) and os.path.exists(waldo_zip):
    print("waldo_kitchen_mask.zip 압축 해제 중...")
    !unzip -q -o {waldo_zip} -d /tmp/_tmp_waldo
    !mv /tmp/_tmp_waldo/waldo_kitchen/gt_mask {waldo_gt}
    !rm -rf /tmp/_tmp_waldo
    print("waldo_kitchen gt_mask 완료!")
elif os.path.exists(waldo_gt):
    print("waldo_kitchen gt_mask 이미 존재.")

# Step 6: 구조 확인
print("\n=== 데이터 구조 확인 ===")
for scene_name in ["figurines", "ramen", "teatime", "waldo_kitchen"]:
    scene_dir = os.path.join(DATASET_DIR, scene_name)
    has_images = os.path.isdir(os.path.join(scene_dir, "images"))
    has_sparse = os.path.exists(os.path.join(scene_dir, "sparse", "0", "cameras.bin"))
    has_json = os.path.isdir(os.path.join(scene_dir, "json"))
    has_gt = os.path.isdir(os.path.join(scene_dir, "gt_mask"))
    ok = lambda x: "OK" if x else "MISSING"
    print(f"  {scene_name}: images={ok(has_images)} sparse={ok(has_sparse)} json={ok(has_json)} gt_mask={ok(has_gt)}")

### 2-1. 데이터 구조 상세 확인
다운받은 데이터의 전체 구조를 확인합니다.

In [ ]:
import os, json

scene = os.path.join(DATA_ROOT, SCENE)

# 1. 전체 폴더 구조 (깊이 3까지)
print("=== 폴더 구조 ===")
!find {scene} -maxdepth 3 -type d | sort

# 2. 각 폴더별 파일 수
print("\n=== 폴더별 파일 수 ===")
for root, dirs, files in os.walk(scene):
    depth = root.replace(scene, '').count(os.sep)
    if depth <= 2:
        indent = '  ' * depth
        print(f"{indent}{os.path.basename(root)}/  ({len(files)} files)")

# 3. 이미지 파일 목록 (처음/끝)
print("\n=== images/ ===")
imgs = sorted(os.listdir(os.path.join(scene, "images")))
print(f"총 {len(imgs)}개")
for f in imgs[:3]: print(f"  {f}")
print("  ...")
for f in imgs[-3:]: print(f"  {f}")

# 4. sparse/0/ 파일
print("\n=== sparse/0/ ===")
!ls -lh {scene}/sparse/0/

# 5. train_json 파일 목록
print("\n=== json/train_json/ ===")
train_dir = os.path.join(scene, "json", "train_json")
if os.path.isdir(train_dir):
    train_jsons = sorted(os.listdir(train_dir))
    print(f"총 {len(train_jsons)}개: {train_jsons[:3]} ... {train_jsons[-3:]}")
else:
    print("없음")

# 6. test_json 파일 목록
print("\n=== json/test_json/ ===")
test_dir = os.path.join(scene, "json", "test_json")
if os.path.isdir(test_dir):
    test_jsons = sorted(os.listdir(test_dir))
    print(f"총 {len(test_jsons)}개: {test_jsons}")
else:
    print("없음")

# 7. train JSON 샘플 (전체 내용)
if os.path.isdir(train_dir) and train_jsons:
    print(f"\n=== train_json 샘플 ({train_jsons[0]}) ===")
    with open(os.path.join(train_dir, train_jsons[0])) as f:
        data = json.load(f)
    print(json.dumps(data, indent=2, ensure_ascii=False))

# 8. test JSON 샘플 (전체 내용)
if os.path.isdir(test_dir) and test_jsons:
    print(f"\n=== test_json 샘플 ({test_jsons[0]}) ===")
    with open(os.path.join(test_dir, test_jsons[0])) as f:
        data = json.load(f)
    print(json.dumps(data, indent=2, ensure_ascii=False))

# 9. gt_mask 구조
gt_mask_dir = os.path.join(scene, "gt_mask")
if os.path.isdir(gt_mask_dir):
    gt_dirs = sorted(os.listdir(gt_mask_dir))
    print(f"\n=== gt_mask/ ===")
    print(f"총 {len(gt_dirs)}개 프레임 폴더: {gt_dirs[:3]} ... {gt_dirs[-3:]}")

    # 첫 번째 프레임의 마스크 파일들
    first_dir = os.path.join(gt_mask_dir, gt_dirs[0])
    print(f"\n=== gt_mask/{gt_dirs[0]}/ ===")
    masks = sorted(os.listdir(first_dir))
    for m in masks:
        size = os.path.getsize(os.path.join(first_dir, m))
        print(f"  {m} ({size} bytes)")

# 10. distorted/ 구조
distorted = os.path.join(scene, "distorted")
if os.path.isdir(distorted):
    print("\n=== distorted/ ===")
    !ls -lh {distorted}/sparse/0/

## 3. 체크포인트 다운로드

사전학습된 3DGS RGB 체크포인트가 필요합니다.

**다운로드:**
- Google Drive: https://drive.google.com/drive/folders/1z9O2FWwUlE29lSgLDj9Af7sv5ZQv6sc_
- HuggingFace: https://huggingface.co/FudanCVL/RefSplat

In [ ]:
# HuggingFace에서 체크포인트 다운로드
!pip install huggingface_hub -q
from huggingface_hub import snapshot_download

if not os.path.exists(os.path.join(CHECKPOINT_ROOT, "ramenchkpnt30000.pth")):
    snapshot_download(
        repo_id="FudanCVL/RefSplat",
        local_dir=CHECKPOINT_ROOT,
    )
    print("체크포인트 다운로드 완료!")
else:
    print("체크포인트 이미 존재. 스킵.")

# 확인
print("\n체크포인트 경로 확인:")
for scene_name, ckpt_name in CKPT_NAME_MAP.items():
    path = os.path.join(CHECKPOINT_ROOT, ckpt_name)
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"  [{status}] {scene_name}: {ckpt_name}")

## 4. 데이터 구조 검증

In [ ]:
import json
from pathlib import Path

def verify_scene(scene_path):
    """데이터셋 구조 검증"""
    checks = {
        "images/": os.path.isdir(os.path.join(scene_path, "images")),
        "sparse/0/": os.path.isdir(os.path.join(scene_path, "sparse", "0")),
        "cameras.bin": os.path.isfile(os.path.join(scene_path, "sparse", "0", "cameras.bin")),
        "images.bin": os.path.isfile(os.path.join(scene_path, "sparse", "0", "images.bin")),
        "points3D.bin": os.path.isfile(os.path.join(scene_path, "sparse", "0", "points3D.bin")),
        "gt_mask/": os.path.isdir(os.path.join(scene_path, "gt_mask")),
    }
    # json 폴더는 json/ 아래 또는 직접
    has_json = os.path.isdir(os.path.join(scene_path, "json")) or \
               os.path.isdir(os.path.join(scene_path, "train_json"))
    checks["json annotations"] = has_json

    print(f"\n--- Scene: {os.path.basename(scene_path)} ---")
    all_ok = True
    for name, ok in checks.items():
        status = "OK" if ok else "MISSING"
        print(f"  [{status}] {name}")
        if not ok:
            all_ok = False

    # 이미지 수 확인
    img_dir = os.path.join(scene_path, "images")
    if os.path.isdir(img_dir):
        n_images = len([f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png'))])
        print(f"  Images: {n_images}")

    # GT mask 수 확인
    mask_dir = os.path.join(scene_path, "gt_mask")
    if os.path.isdir(mask_dir):
        n_masks = len([f for f in os.listdir(mask_dir) if f.endswith('.png')])
        print(f"  GT masks: {n_masks}")

    # 샘플 JSON 확인
    json_dir = os.path.join(scene_path, "json", "train_json")
    if not os.path.isdir(json_dir):
        json_dir = os.path.join(scene_path, "train_json")
    if os.path.isdir(json_dir):
        json_files = sorted([f for f in os.listdir(json_dir) if f.endswith('.json')])
        print(f"  Train JSONs: {len(json_files)}")
        if json_files:
            with open(os.path.join(json_dir, json_files[0])) as f:
                sample = json.load(f)
            print(f"  Sample JSON keys: {list(sample.keys())}")
            if 'object' in sample:
                obj = sample['object'][0]
                print(f"  Sample object keys: {list(obj.keys())}")
                if 'sentence' in obj:
                    print(f"  Sample sentences: {obj['sentence'][:2]}")

    return all_ok

# 선택한 장면 검증
if os.path.exists(SCENE_DATA_PATH):
    verify_scene(SCENE_DATA_PATH)
else:
    print(f"데이터 경로가 존재하지 않습니다: {SCENE_DATA_PATH}")

## 5. ReferSplat 학습 (Train)

**학습 흐름:**
1. 사전학습된 3DGS RGB 체크포인트 로드 (RGB 파라미터 freeze)
2. 각 Gaussian에 16차원 referring feature 추가
3. BERT로 텍스트 임베딩 → Position-aware Cross-Modal Interaction
4. BCE Loss + Contrastive Loss로 학습
5. 5 epoch, ~58분 (A6000 기준)

**주요 하이퍼파라미터:**
- referring feature dim: 16
- language_feature_lr: 0.0025
- mlp_lr / cross_attention_lr: 0.0001
- contrastive loss weight: 0.1
- top-τ schedule: 0.1 × 0.6^(iter/1000)

In [ ]:
%cd {REPO_DIR}
import glob

if USE_AUTHOR_MODEL:
    author_ckpt = os.path.join(CHECKPOINT_ROOT, AUTHOR_CKPT_MAP.get(SCENE, ""))
    if os.path.exists(author_ckpt):
        print(f"저자 제공 모델 사용: {author_ckpt}")
        print("학습을 건너뜁니다. 바로 렌더링으로 이동하세요.")
    else:
        print(f"[NOT FOUND] 저자 모델: {author_ckpt}")
        print("직접 학습합니다.")
        USE_AUTHOR_MODEL = False

if not USE_AUTHOR_MODEL:
    # 직접 학습
    existing_ckpts = sorted(glob.glob(os.path.join(SCENE_OUTPUT_PATH, "chkpnt*.pth")))
    if existing_ckpts:
        print(f"학습된 체크포인트 발견! 학습을 건너뜁니다: {os.path.basename(existing_ckpts[-1])}")
    else:
        print(f"체크포인트 없음. 학습을 시작합니다.")
        !python train.py \
            -s {SCENE_DATA_PATH} \
            -m {SCENE_OUTPUT_PATH} \
            --start_checkpoint {PRETRAINED_CKPT}

# 평가에 사용할 출력 경로 설정
if USE_AUTHOR_MODEL:
    EVAL_OUTPUT_PATH = os.path.join(OUTPUT_ROOT, SCENE + "_author")
else:
    EVAL_OUTPUT_PATH = SCENE_OUTPUT_PATH

In [ ]:
# 학습 완료 후 체크포인트를 Google Drive에 저장
import glob, shutil

if USE_AUTHOR_MODEL:
    print("저자 모델 사용 모드 - Drive 저장 불필요")
else:
    ckpt_files = sorted(glob.glob(os.path.join(SCENE_OUTPUT_PATH, "chkpnt*.pth")))
    if ckpt_files:
        drive_scene_dir = os.path.join(DRIVE_SAVE_DIR, SCENE)
        os.makedirs(drive_scene_dir, exist_ok=True)
        for ckpt in ckpt_files:
            dst = os.path.join(drive_scene_dir, os.path.basename(ckpt))
            if not os.path.exists(dst):
                print(f"Drive에 저장 중: {os.path.basename(ckpt)}")
                shutil.copy2(ckpt, dst)
        print(f"저장 완료! 경로: {drive_scene_dir}")
        print("다음 세션에서 자동으로 복원됩니다.")
    else:
        print("저장할 체크포인트가 없습니다.")

### 5-1. (선택) 학습 모니터링
저자 모델 사용 시 이 섹션은 건너뛰어도 됩니다.

### 5-1. 학습 설정 확인
실제 학습에 사용된 하이퍼파라미터와 iteration 수를 확인합니다.

In [ ]:
# [저자 모델 사용 시 스킵] 학습 설정 확인은 직접 학습할 때만 필요합니다.
# 아래 코드를 실행하려면 주석을 해제하세요.

# import os, json, glob, torch

# # --- 1. 코드 하이퍼파라미터 ---
# print("=" * 60)
# print("학습 하이퍼파라미터 (코드 기준)")
# print("=" * 60)
# print(f"  epoch_num:              5")
# print(f"  language_feature_lr:    0.0025")
# print(f"  mlp_lr:                 0.0001")
# print(f"  cross_attention_lr:     0.0001")
# print(f"  referring feature dim:  16")
# print(f"  shared feature dim:     128")
# print(f"  contrastive weight:     0.1")
# print(f"  tau init:               0.1")
# print(f"  tau decay:              ×0.6 every 2000 iter (min 0.005)")
# print(f"  loss:                   BCE + 0.1 × contrastive")

# # --- 2. 데이터셋 문장 수 (= iteration/epoch 계산) ---
# print(f"\n{'=' * 60}")
# print("데이터셋 문장 수 & Iteration 계산")
# print("=" * 60)

# for scene_name in ["ramen", "figurines", "teatime", "waldo_kitchen"]:
#     scene_data = os.path.join(DATA_ROOT, scene_name)
#     # train json
#     json_dir = os.path.join(scene_data, "json", "train_json")
#     if not os.path.isdir(json_dir):
#         json_dir = os.path.join(scene_data, "train_json")
#     if not os.path.isdir(json_dir):
#         print(f"  {scene_name}: train_json 없음")
#         continue
#     total_sentences = 0
#     n_frames = 0
#     for jf in sorted(glob.glob(os.path.join(json_dir, "*.json"))):
#         with open(jf) as f:
#             data = json.load(f)
#         for obj in data.get("object", []):
#             total_sentences += len(obj.get("sentence", []))
#         n_frames += 1
#     iters_5 = total_sentences * 5
#     print(f"  {scene_name:<20} train frames: {n_frames:<5} sentences/epoch: {total_sentences:<6} 5 epochs: {iters_5} iterations")

# # --- 3. 체크포인트 실제 iteration 수 ---
# print(f"\n{'=' * 60}")
# print("체크포인트 저장된 iteration 수")
# print("=" * 60)

# for scene_name in ["ramen", "figurines", "teatime", "waldo_kitchen"]:
#     scene_output = os.path.join(OUTPUT_ROOT, scene_name)
#     ckpts = sorted(glob.glob(os.path.join(scene_output, "chkpnt*.pth")))
#     if ckpts:
#         for ckpt_path in ckpts:
#             try:
#                 _, iteration = torch.load(ckpt_path, map_location="cpu", weights_only=False)
#                 print(f"  {scene_name:<20} {os.path.basename(ckpt_path):<35} iteration: {iteration}")
#             except Exception as e:
#                 print(f"  {scene_name:<20} {os.path.basename(ckpt_path):<35} 로드 실패: {e}")
#     else:
#         print(f"  {scene_name:<20} 체크포인트 없음")

In [ ]:
# 저장된 체크포인트 확인
import glob

if USE_AUTHOR_MODEL:
    author_ckpt = os.path.join(CHECKPOINT_ROOT, AUTHOR_CKPT_MAP.get(SCENE, ""))
    print(f"사용 중인 모델: 저자 제공 ({os.path.basename(author_ckpt)})")
    if os.path.exists(author_ckpt):
        size_mb = os.path.getsize(author_ckpt) / (1024 * 1024)
        print(f"  {os.path.basename(author_ckpt)} ({size_mb:.1f} MB)")
else:
    ckpt_files = sorted(glob.glob(os.path.join(SCENE_OUTPUT_PATH, "chkpnt*.pth")))
    print(f"사용 중인 모델: 직접 학습 ({len(ckpt_files)}개 체크포인트)")
    for f in ckpt_files:
        size_mb = os.path.getsize(f) / (1024 * 1024)
        print(f"  {os.path.basename(f)} ({size_mb:.1f} MB)")

## 6. 렌더링 (Render)

학습된 모델로 테스트 뷰에서 세그멘테이션 마스크를 렌더링합니다.

**테스트 프레임 인덱스:**
- ramen: [6, 24, 60, 65, 81, 119, 128]
- figurines: [83, 97, 146, 179]
- teatime: [2, 25, 43, 107, 129, 140]
- waldo: [19, 35, 67, 105, 162]

In [ ]:
%%writefile {REPO_DIR}/render_colab.py
"""
Colab용 렌더링 스크립트 - 경로를 인자로 받음
"""
import re
import numpy as np
import torch
from scene import Scene
import os
from tqdm import tqdm
from os import makedirs
import torch.nn.functional as F
from gaussian_renderer import render
import torchvision
import random
from utils.general_utils import safe_state
from argparse import ArgumentParser
from arguments import ModelParams, PipelineParams, OptimizationParams
from gaussian_renderer import GaussianModel


def render_set(model_path, source_path, name, views, gaussians, pipeline, background, args):
    render_path = os.path.join(model_path, name, "renders")
    gts_path = os.path.join(model_path, name, "gt")
    render_npy_path = os.path.join(model_path, name, "renders_npy")
    gts_npy_path = os.path.join(model_path, name, "gt_npy")

    makedirs(render_npy_path, exist_ok=True)
    makedirs(gts_npy_path, exist_ok=True)
    makedirs(render_path, exist_ok=True)
    makedirs(gts_path, exist_ok=True)

    for idx, view in enumerate(tqdm(views, desc="Rendering progress")):
        for i in range(len(view.sentence)):
            sn = view.image_name
            number = re.findall(r'\d+', sn)
            number_int = int(number[0])
            output = render(view, gaussians, pipeline, background, args, sentence=view.sentence[i])
            rendering = output["language_feature_image"]
            rendering = torch.sigmoid(rendering)
            rendering = (rendering >= 0.5).float()
            gt = view.gt_mask[view.category[i]]
            fname = '{0:05d}'.format(number_int) + '{}'.format(view.category[i])
            np.save(os.path.join(render_npy_path, fname + ".npy"), rendering.permute(1, 2, 0).cpu().numpy())
            np.save(os.path.join(gts_npy_path, fname + ".npy"), gt.permute(1, 2, 0).cpu().numpy())
            torchvision.utils.save_image(rendering, os.path.join(render_path, fname + ".png"))
            torchvision.utils.save_image(gt, os.path.join(gts_path, fname + ".png"))


if __name__ == "__main__":
    random.seed(0)
    np.random.seed(0)
    torch.manual_seed(0)
    torch.cuda.manual_seed_all(0)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    parser = ArgumentParser()
    lp = ModelParams(parser)
    op = OptimizationParams(parser)
    pp = PipelineParams(parser)
    parser.add_argument("--checkpoint_file", type=str, required=True)
    parser.add_argument("--quiet", action="store_true")
    args = parser.parse_args()
    args.include_feature = True

    safe_state(args.quiet)

    with torch.no_grad():
        dataset = lp.extract(args)
        gaussians = GaussianModel(dataset.sh_degree)
        scene = Scene(dataset, gaussians, shuffle=False)

        (model_params, first_iter) = torch.load(args.checkpoint_file, map_location=f'cuda:{torch.cuda.current_device()}', weights_only=False)
        gaussians.restore(model_params, args, mode='test')

        bg_color = [1, 1, 1] if dataset.white_background else [0, 0, 0]
        background = torch.tensor(bg_color, dtype=torch.float32, device="cuda")

        render_set(dataset.model_path, dataset.source_path, "test_results",
                   scene.getTestCameras(), gaussians, pp.extract(args), background, args)

    print("Rendering complete!")

In [ ]:
%cd {REPO_DIR}
import glob

# 사용할 체크포인트 결정
if USE_AUTHOR_MODEL:
    ckpt_path = os.path.join(CHECKPOINT_ROOT, AUTHOR_CKPT_MAP[SCENE])
    output_dir = os.path.join(OUTPUT_ROOT, SCENE + "_author")
    print(f"저자 모델로 렌더링: {ckpt_path}")
else:
    ckpt_files = sorted(glob.glob(os.path.join(SCENE_OUTPUT_PATH, "chkpnt*.pth")))
    ckpt_path = ckpt_files[-1] if ckpt_files else None
    output_dir = SCENE_OUTPUT_PATH
    if ckpt_path:
        print(f"직접 학습 모델로 렌더링: {ckpt_path}")
    else:
        print("체크포인트가 없습니다. 먼저 학습을 실행하세요.")

if ckpt_path and os.path.exists(ckpt_path):
    os.makedirs(output_dir, exist_ok=True)
    !python render_colab.py \
        -s {SCENE_DATA_PATH} \
        -m {output_dir} \
        --checkpoint_file {ckpt_path}

## 7. 평가 (mIoU)

In [ ]:
import os, json, torch, numpy as np
from PIL import Image
from collections import defaultdict

# ============================================================
# 전체 결과 한 번에 계산 (이후 모든 실험이 이 데이터를 재사용)
# ============================================================

def load_all_results(data_path, eval_output, scene):
    """렌더링 결과 + GT 마스크를 전부 로드하여 통합 데이터프레임 생성"""
    render_dir = os.path.join(eval_output, "test_results", "renders")
    gt_dir = os.path.join(eval_output, "test_results", "gt")

    if not os.path.exists(render_dir) or not os.path.exists(gt_dir):
        print(f"렌더링 결과 없음: {render_dir}")
        return []

    # test_json에서 문장 로드
    cat_sentences = defaultdict(list)
    json_dir = os.path.join(data_path, "json", "test_json")
    if not os.path.isdir(json_dir):
        json_dir = os.path.join(data_path, "test_json")
    if os.path.isdir(json_dir):
        for jf in sorted(os.listdir(json_dir)):
            if not jf.endswith(".json"):
                continue
            with open(os.path.join(json_dir, jf)) as f:
                data = json.load(f)
            for obj in data.get("object", []):
                cat = obj.get("category", "")
                for sent in obj.get("sentence", []):
                    if sent not in cat_sentences[cat]:
                        cat_sentences[cat].append(sent)

    results = []
    for fname in sorted(os.listdir(render_dir)):
        if not fname.endswith(".png"):
            continue
        gt_path = os.path.join(gt_dir, fname)
        if not os.path.exists(gt_path):
            continue

        pred = torch.from_numpy(np.array(Image.open(os.path.join(render_dir, fname)).convert("L"))).float() > 0
        gt = torch.from_numpy(np.array(Image.open(gt_path).convert("L"))).float() > 0

        gt_area = gt.sum().item()
        total_pixels = gt.numel()
        if gt_area == 0:
            continue

        intersection = torch.logical_and(pred, gt).sum().float().item()
        union = torch.logical_or(pred, gt).sum().float().item()
        pred_area = pred.sum().item()

        iou = intersection / union * 100 if union > 0 else 0.0
        precision = intersection / pred_area * 100 if pred_area > 0 else 0.0
        recall = intersection / gt_area * 100 if gt_area > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        detected = 1 if intersection > 0 else 0

        # 메타 정보 추출
        frame_num = int(''.join([c for c in fname.split('.')[0] if c.isdigit()]))
        category = ''.join([c for c in fname.split('.png')[0] if not c.isdigit()]).strip()
        area_ratio = gt_area / total_pixels * 100
        sentences = cat_sentences.get(category, [])

        results.append({
            "fname": fname, "frame": frame_num, "category": category,
            "iou": iou, "precision": precision, "recall": recall, "f1": f1,
            "detected": detected,
            "gt_area": gt_area, "pred_area": pred_area, "area_ratio": area_ratio,
            "sentences": sentences,
            "pred_path": os.path.join(render_dir, fname),
            "gt_path": gt_path,
        })

    return results

# === 실행 ===
ALL_RESULTS = load_all_results(
    os.path.join(DATA_ROOT, SCENE), EVAL_OUTPUT_PATH, SCENE
)

if ALL_RESULTS:
    ious = [r["iou"] for r in ALL_RESULTS]
    precisions = [r["precision"] for r in ALL_RESULTS]
    recalls = [r["recall"] for r in ALL_RESULTS]
    f1s = [r["f1"] for r in ALL_RESULTS]
    detected = [r["detected"] for r in ALL_RESULTS]

    print(f"\n{'=' * 70}")
    print(f"실험 1: 전체 성능 (Scene: {SCENE}, 저자 모델)")
    print(f"{'=' * 70}")
    print(f"  mIoU:       {np.mean(ious):.2f}%")
    print(f"  Precision:  {np.mean(precisions):.2f}%")
    print(f"  Recall:     {np.mean(recalls):.2f}%")
    print(f"  F1:         {np.mean(f1s):.2f}%")
    print(f"  탐지율:     {np.mean(detected)*100:.1f}% ({sum(detected)}/{len(detected)})")
    print(f"  총 쿼리 수: {len(ALL_RESULTS)}")

    # 객체별 성능
    print(f"\n{'카테고리':<20} {'mIoU':>6} {'Prec':>6} {'Rec':>6} {'F1':>6} {'탐지':>5} {'n':>3}")
    print("-" * 60)
    cats = sorted(set(r["category"] for r in ALL_RESULTS))
    for cat in cats:
        cr = [r for r in ALL_RESULTS if r["category"] == cat]
        print(f"  {cat:<18} {np.mean([r['iou'] for r in cr]):>5.1f}% "
              f"{np.mean([r['precision'] for r in cr]):>5.1f}% "
              f"{np.mean([r['recall'] for r in cr]):>5.1f}% "
              f"{np.mean([r['f1'] for r in cr]):>5.1f}% "
              f"{sum(r['detected'] for r in cr)}/{len(cr):>2}")

    # 문장과 함께 상세 출력
    print(f"\n{'File':<35} {'IoU':>6} {'Prec':>6} {'Rec':>6}  Sentence")
    print("-" * 100)
    for r in ALL_RESULTS:
        sent = r["sentences"][0][:50] + "..." if r["sentences"] else "-"
        print(f"  {r['fname']:<33} {r['iou']:>5.1f}% {r['precision']:>5.1f}% {r['recall']:>5.1f}%  {sent}")
else:
    print("결과 없음. 렌더링을 먼저 실행하세요.")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import numpy as np
import os, glob

def visualize_results_with_rgb(data_path, render_dir, gt_dir, n_samples=6):
    """RGB 원본 + Predicted mask + GT mask + 오버레이 시각화"""
    render_files = sorted([f for f in os.listdir(render_dir) if f.endswith('.png')])
    n_show = min(n_samples, len(render_files))

    if n_show == 0:
        print("시각화할 결과가 없습니다.")
        return

    # 이미지 폴더
    img_dir = os.path.join(data_path, "images")

    fig, axes = plt.subplots(n_show, 4, figsize=(20, 4 * n_show))
    if n_show == 1:
        axes = axes[np.newaxis, :]

    for i, fname in enumerate(render_files[:n_show]):
        # 프레임 번호 추출 → RGB 이미지 찾기
        frame_num = int(''.join([c for c in fname.split('.')[0] if c.isdigit()]))
        obj_name = ''.join([c for c in fname.split('.png')[0] if not c.isdigit()]).strip()

        # RGB 이미지 찾기 (frame_00006.jpg 등)
        rgb_path = os.path.join(img_dir, f"frame_{frame_num:05d}.jpg")
        if not os.path.exists(rgb_path):
            rgb_path = os.path.join(img_dir, f"frame_{frame_num:05d}.png")

        # 이미지 로드
        pred = np.array(Image.open(os.path.join(render_dir, fname)).convert("L"))
        gt_path = os.path.join(gt_dir, fname)
        gt = np.array(Image.open(gt_path).convert("L")) if os.path.exists(gt_path) else np.zeros_like(pred)

        # IoU 계산
        pred_bin = pred > 0
        gt_bin = gt > 0
        intersection = np.logical_and(pred_bin, gt_bin).sum()
        union = np.logical_or(pred_bin, gt_bin).sum()
        iou = intersection / union * 100 if union > 0 else 0.0

        # 1) RGB 원본
        if os.path.exists(rgb_path):
            rgb = np.array(Image.open(rgb_path))
            axes[i, 0].imshow(rgb)
        else:
            axes[i, 0].text(0.5, 0.5, "RGB not found", ha='center', va='center', transform=axes[i, 0].transAxes)
        axes[i, 0].set_title(f"RGB (frame {frame_num})", fontsize=10)
        axes[i, 0].axis('off')

        # 2) RGB + 오버레이 (Pred=빨강, GT=초록, 겹침=노랑)
        if os.path.exists(rgb_path):
            rgb_overlay = rgb.copy().astype(float)
            # GT: 초록 오버레이
            gt_mask_3d = np.stack([np.zeros_like(gt_bin), gt_bin, np.zeros_like(gt_bin)], axis=-1).astype(float)
            # Pred: 빨강 오버레이
            pred_mask_3d = np.stack([pred_bin, np.zeros_like(pred_bin), np.zeros_like(pred_bin)], axis=-1).astype(float)
            overlay = rgb_overlay / 255.0
            overlay = overlay * 0.6 + gt_mask_3d * 0.4
            overlay = np.clip(overlay + pred_mask_3d * 0.3, 0, 1)
            axes[i, 1].imshow(overlay)
            # 범례
            legend_patches = [
                mpatches.Patch(color='green', alpha=0.5, label='GT'),
                mpatches.Patch(color='red', alpha=0.5, label='Pred'),
            ]
            axes[i, 1].legend(handles=legend_patches, loc='upper right', fontsize=7)
        axes[i, 1].set_title(f"Overlay: {obj_name} (IoU: {iou:.1f}%)", fontsize=10)
        axes[i, 1].axis('off')

        # 3) Predicted mask
        axes[i, 2].imshow(pred, cmap='gray')
        axes[i, 2].set_title(f"Predicted", fontsize=10)
        axes[i, 2].axis('off')

        # 4) GT mask
        axes[i, 3].imshow(gt, cmap='gray')
        axes[i, 3].set_title(f"Ground Truth", fontsize=10)
        axes[i, 3].axis('off')

    plt.tight_layout()
    plt.show()

# 실행
for scene in ALL_SCENES:
    scene_data = os.path.join(DATA_ROOT, scene)
    scene_output = os.path.join(OUTPUT_ROOT, scene + "_author") if USE_AUTHOR_MODEL else os.path.join(OUTPUT_ROOT, scene)
    r_dir = os.path.join(scene_output, "test_results", "renders")
    g_dir = os.path.join(scene_output, "test_results", "gt")

    if os.path.exists(r_dir):
        print(f"\n{'='*60}")
        print(f"Scene: {scene}")
        print(f"{'='*60}")
        visualize_results_with_rgb(scene_data, r_dir, g_dir, n_samples=8)
    else:
        print(f"{scene}: 렌더링 결과 없음")

## 8. 시각화

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

def visualize_results(render_dir, gt_dir, n_samples=4):
    """렌더링 결과 시각화 (Predicted vs GT)"""
    render_files = sorted([f for f in os.listdir(render_dir) if f.endswith('.png')])
    n_show = min(n_samples, len(render_files))

    if n_show == 0:
        print("시각화할 결과가 없습니다.")
        return

    fig, axes = plt.subplots(n_show, 2, figsize=(10, 4 * n_show))
    if n_show == 1:
        axes = axes[np.newaxis, :]

    for i, fname in enumerate(render_files[:n_show]):
        pred = np.array(Image.open(os.path.join(render_dir, fname)).convert("L"))
        gt_path = os.path.join(gt_dir, fname)
        gt = np.array(Image.open(gt_path).convert("L")) if os.path.exists(gt_path) else np.zeros_like(pred)

        axes[i, 0].imshow(pred, cmap='gray')
        axes[i, 0].set_title(f"Predicted: {fname}")
        axes[i, 0].axis('off')

        axes[i, 1].imshow(gt, cmap='gray')
        axes[i, 1].set_title(f"Ground Truth: {fname}")
        axes[i, 1].axis('off')

    plt.tight_layout()
    plt.show()

render_dir = os.path.join(EVAL_OUTPUT_PATH, "test_results", "renders")
gt_dir = os.path.join(EVAL_OUTPUT_PATH, "test_results", "gt")

if os.path.exists(render_dir):
    visualize_results(render_dir, gt_dir)
else:
    print("렌더링 결과가 없습니다.")

## 9. (선택) Two-Stage 최적화
**저자 모델 사용 시 스킵**

Stage 1에서 렌더링된 마스크를 새로운 pseudo GT로 사용하여 재학습합니다.
논문 기준 ramen: 35.2 → 36.9 mIoU 향상.

In [ ]:
# [저자 모델 사용 시 스킵] 섹션 5-7에서 이미 렌더링/평가 완료
# 직접 학습하려면 아래 주석을 해제하세요.

# # 전체 장면 일괄 학습 + 렌더링 + 평가
# # gt_mask가 제공된 장면만 실행

# import glob, os, numpy as np

# ALL_SCENES = ["ramen"]
# # ALL_SCENES = ["ramen", "waldo_kitchen"]  # waldo_kitchen은 학습 미완료

# CKPT_NAME_MAP = {
#     "figurines": "figurineschkpnt30000.pth",
#     "ramen": "ramenchkpnt30000.pth",
#     "teatime": "teatimechkpnt30000.pth",
#     "waldo_kitchen": "kitchenchkpnt30000.pth",
# }

# results = {}

# for scene in ALL_SCENES:
#     print(f"\n{'='*60}")
#     print(f"Processing scene: {scene}")
#     print(f"{'='*60}")

#     scene_data = os.path.join(DATA_ROOT, scene)
#     scene_output = os.path.join(OUTPUT_ROOT, scene + "_author") if USE_AUTHOR_MODEL else os.path.join(OUTPUT_ROOT, scene)
#     scene_ckpt = os.path.join(CHECKPOINT_ROOT, CKPT_NAME_MAP[scene])

#     # 사전학습 체크포인트 확인
#     if not os.path.exists(scene_ckpt):
#         print(f"  [SKIP] 사전학습 체크포인트 없음: {scene_ckpt}")
#         print(f"  섹션 3에서 체크포인트를 다시 다운로드하세요.")
#         continue

#     # 학습 (체크포인트 없으면)
#     existing = sorted(glob.glob(os.path.join(scene_output, "chkpnt*.pth")))
#     if existing:
#         print(f"  체크포인트 발견, 학습 스킵: {os.path.basename(existing[-1])}")
#     else:
#         os.makedirs(scene_output, exist_ok=True)
#         !cd {REPO_DIR} && python train.py -s {scene_data} -m {scene_output} --start_checkpoint {scene_ckpt}

#     # 렌더링
#     ckpt_files = sorted(glob.glob(os.path.join(scene_output, "chkpnt*.pth")))
#     if ckpt_files:
#         latest = ckpt_files[-1]
#         !cd {REPO_DIR} && python render_colab.py -s {scene_data} -m {scene_output} --checkpoint_file {latest}

#     # 평가
#     r_dir = os.path.join(scene_output, "test_results", "renders")
#     g_dir = os.path.join(scene_output, "test_results", "gt")
#     if os.path.exists(r_dir) and os.path.exists(g_dir):
#         miou, _ = evaluate_miou(r_dir, g_dir)
#         results[scene] = miou
#         print(f"  {scene} mIoU: {miou*100:.2f}%")
#     else:
#         print(f"  {scene} 렌더링 결과 없음")

# # 최종 결과
# if results:
#     avg = np.mean(list(results.values()))
#     print(f"\n{'='*60}")
#     print("Final Results:")
#     for s, v in results.items():
#         print(f"  {s}: {v*100:.2f}%")
#     print(f"  Average: {avg*100:.2f}%")
#     print(f"{'='*60}")

## 10. 전체 장면 일괄 실행
**저자 모델 사용 시 스킵** (섹션 5-7에서 이미 처리됨)

4개 장면 모두 학습 + 평가하려면 아래를 실행하세요.

---
## 11. 문제점 분석 실험

### 11-1. 객체 크기별 성능 분석
GT 마스크의 면적(픽셀 수) 대비 IoU를 분석하여, 작은 객체일수록 성능이 떨어지는지 정량적으로 확인합니다.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ============================================================
# 실험 2: 객체 크기 vs IoU (ALL_RESULTS 재사용)
# ============================================================
if ALL_RESULTS:
    sizes = [r["area_ratio"] for r in ALL_RESULTS]
    ious = [r["iou"] for r in ALL_RESULTS]
    detects = [r["detected"] for r in ALL_RESULTS]
    labels = [r["category"] for r in ALL_RESULTS]

    sizes_arr = np.array(sizes)
    ious_arr = np.array(ious)
    detects_arr = np.array(detects)

    # Scatter plot
    fig, ax = plt.subplots(1, 1, figsize=(12, 7))
    scatter = ax.scatter(sizes, ious, c=ious, cmap='RdYlGn', s=60, alpha=0.7, edgecolors='black', linewidth=0.5)
    plt.colorbar(scatter, label='IoU (%)')
    for label in set(labels):
        idxs = [i for i, l in enumerate(labels) if l == label]
        ax.annotate(label, (np.mean([sizes[i] for i in idxs]), np.mean([ious[i] for i in idxs])),
                    fontsize=8, ha='center', bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))
    ax.set_xlabel('GT Mask Area (% of image)', fontsize=12)
    ax.set_ylabel('IoU (%)', fontsize=12)
    ax.set_title('Object Size vs Segmentation IoU', fontsize=14)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # 크기 구간별 통계 (IoU + 탐지율)
    bins = [(0, 1, "Tiny (<1%)"), (1, 5, "Small (1-5%)"), (5, 15, "Medium (5-15%)"), (15, 100, "Large (>15%)")]
    print(f"\n{'구간':<20} {'n':<5} {'mIoU':<10} {'std':<10} {'탐지율':<10}")
    print("-" * 55)
    for lo, hi, name in bins:
        mask = (sizes_arr >= lo) & (sizes_arr < hi)
        if mask.sum() > 0:
            print(f"{name:<20} {mask.sum():<5} {ious_arr[mask].mean():<10.1f} {ious_arr[mask].std():<10.1f} {detects_arr[mask].mean()*100:<10.1f}%")

    corr = np.corrcoef(sizes_arr, ious_arr)[0, 1]
    print(f"\n면적-IoU Pearson 상관계수: {corr:.3f}")
else:
    print("ALL_RESULTS 없음. 실험 1 셀을 먼저 실행하세요.")

### 11-2. 객체 유형별 성능 분석 (투명/반사/일반)
투명 객체, 반사 객체, 일반 객체로 분류하여 3DGS 기반 방법의 근본적 한계를 분석합니다.

In [ ]:
import numpy as np, matplotlib.pyplot as plt

# ============================================================
# 실험 3: 객체 속성별 성능 (ALL_RESULTS 재사용)
# ============================================================
OBJECT_TYPE = {
    "glass of water": "transparent", "sake cup": "transparent",
    "chopsticks": "thin", "spoon": "thin",
    "bowl": "opaque", "plate": "opaque", "nori": "opaque",
    "egg": "opaque", "eggs": "opaque", "corn": "opaque",
    "kamaboko": "opaque", "wavy noodles": "opaque", "hand": "opaque",
}

if ALL_RESULTS:
    type_data = {"transparent": [], "thin": [], "opaque": []}
    for r in ALL_RESULTS:
        t = OBJECT_TYPE.get(r["category"], "opaque")
        type_data[t].append(r)

    types_with_data = {k: v for k, v in type_data.items() if v}
    type_names = list(types_with_data.keys())
    type_means = [np.mean([r["iou"] for r in types_with_data[t]]) for t in type_names]
    type_stds = [np.std([r["iou"] for r in types_with_data[t]]) for t in type_names]
    type_counts = [len(types_with_data[t]) for t in type_names]
    type_detect = [np.mean([r["detected"] for r in types_with_data[t]])*100 for t in type_names]

    colors_map = {"transparent": "#3498db", "thin": "#e74c3c", "opaque": "#2ecc71"}
    bar_colors = [colors_map.get(t, "#95a5a6") for t in type_names]

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.bar(type_names, type_means, yerr=type_stds, capsize=5, color=bar_colors, edgecolor='black', alpha=0.8)
    for bar, count, mean, det in zip(bars, type_counts, type_means, type_detect):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 2,
                f'n={count}\n{mean:.1f}%\ndet:{det:.0f}%', ha='center', va='bottom', fontsize=9)
    ax.set_ylabel('Mean IoU (%)', fontsize=12)
    ax.set_title('Segmentation Performance by Object Type', fontsize=14)
    ax.set_ylim(0, 100)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f"\n{'유형':<15} {'n':<5} {'mIoU':<10} {'std':<10} {'탐지율':<10} {'Precision':<10} {'Recall':<10}")
    print("-" * 70)
    for t in type_names:
        v = types_with_data[t]
        print(f"{t:<15} {len(v):<5} {np.mean([r['iou'] for r in v]):<10.1f} "
              f"{np.std([r['iou'] for r in v]):<10.1f} "
              f"{np.mean([r['detected'] for r in v])*100:<10.1f} "
              f"{np.mean([r['precision'] for r in v]):<10.1f} "
              f"{np.mean([r['recall'] for r in v]):<10.1f}")
else:
    print("ALL_RESULTS 없음.")

### 11-3. Referring Expression 유형별 성능 분석
JSON 어노테이션의 sentence를 분석하여, 공간 관계 표현(spatial)과 속성 표현(attribute)에 따른 성능 차이를 확인합니다.
- **Spatial**: "next to", "behind", "between", "left", "right", "near", "placed" 등
- **Attribute**: "red", "round", "small", "large", "colored" 등
- **Mixed**: 두 가지 모두 포함

In [ ]:
import numpy as np, matplotlib.pyplot as plt

# ============================================================
# 실험 4: Referring Expression 유형별 성능 (ALL_RESULTS 재사용)
# ============================================================
SPATIAL_KEYWORDS = [
    "next to", "behind", "in front of", "between", "left", "right",
    "near", "close to", "placed", "above", "below", "on top of",
    "beside", "adjacent", "opposite", "facing", "under", "over",
    "resting on", "sitting on", "standing on"
]
ATTRIBUTE_KEYWORDS = [
    "red", "blue", "green", "yellow", "white", "black", "brown", "orange", "pink",
    "round", "square", "small", "large", "big", "tiny", "long", "short", "thin",
    "colored", "bright", "dark", "shiny", "surface", "flat", "wooden", "metal",
    "transparent", "glass"
]

def classify_sentence(sentence):
    s = sentence.lower()
    has_spatial = any(kw in s for kw in SPATIAL_KEYWORDS)
    has_attribute = any(kw in s for kw in ATTRIBUTE_KEYWORDS)
    if has_spatial and has_attribute: return "mixed"
    elif has_spatial: return "spatial"
    elif has_attribute: return "attribute"
    else: return "simple"

if ALL_RESULTS:
    type_ious = {"spatial": [], "attribute": [], "mixed": [], "simple": []}
    for r in ALL_RESULTS:
        if r["sentences"]:
            types = [classify_sentence(s) for s in r["sentences"]]
            dominant = max(set(types), key=types.count)
        else:
            dominant = "simple"
        type_ious[dominant].append(r)

    # 바 차트
    types_with_data = {k: v for k, v in type_ious.items() if v}
    type_names = list(types_with_data.keys())
    type_means = [np.mean([r["iou"] for r in types_with_data[t]]) for t in type_names]
    type_stds = [np.std([r["iou"] for r in types_with_data[t]]) for t in type_names]
    type_counts = [len(types_with_data[t]) for t in type_names]

    colors_map = {"spatial": "#e74c3c", "attribute": "#3498db", "mixed": "#9b59b6", "simple": "#2ecc71"}
    bar_colors = [colors_map.get(t, "#95a5a6") for t in type_names]

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.bar(type_names, type_means, yerr=type_stds, capsize=5, color=bar_colors, edgecolor='black', alpha=0.8)
    for bar, count, mean in zip(bars, type_counts, type_means):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 2,
                f'n={count}\n{mean:.1f}%', ha='center', va='bottom', fontsize=10)
    ax.set_ylabel('Mean IoU (%)', fontsize=12)
    ax.set_title('Performance by Referring Expression Type', fontsize=14)
    ax.set_ylim(0, 100)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

    # 테이블
    print(f"\n{'유형':<12} {'n':<5} {'mIoU':<8} {'Prec':<8} {'Rec':<8} {'탐지율':<8}")
    print("-" * 50)
    for t in type_names:
        v = types_with_data[t]
        print(f"{t:<12} {len(v):<5} {np.mean([r['iou'] for r in v]):<8.1f} "
              f"{np.mean([r['precision'] for r in v]):<8.1f} "
              f"{np.mean([r['recall'] for r in v]):<8.1f} "
              f"{np.mean([r['detected'] for r in v])*100:<8.1f}")

    # 문장 길이 vs IoU 추가 분석
    sent_lens = []
    sent_ious = []
    for r in ALL_RESULTS:
        if r["sentences"]:
            avg_len = np.mean([len(s.split()) for s in r["sentences"]])
            sent_lens.append(avg_len)
            sent_ious.append(r["iou"])

    if sent_lens:
        corr = np.corrcoef(sent_lens, sent_ious)[0, 1]
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.scatter(sent_lens, sent_ious, c=sent_ious, cmap='RdYlGn', s=50, edgecolors='black', alpha=0.7)
        ax.set_xlabel('Avg Sentence Length (words)', fontsize=12)
        ax.set_ylabel('IoU (%)', fontsize=12)
        ax.set_title(f'Sentence Length vs IoU (r={corr:.3f})', fontsize=14)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        print(f"\n문장길이-IoU 상관계수: {corr:.3f}")
else:
    print("ALL_RESULTS 없음.")

#### Referring Expression 유형별 결과 이미지
각 유형(spatial/attribute/mixed/simple)별로 추론 결과를 RGB 오버레이와 함께 시각화합니다.

In [ ]:
import os, json, re, torch, numpy as np, matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from collections import defaultdict

def visualize_expression_types(data_root, output_root, scene, n_per_type=3):
    """유형별 대표 결과를 RGB 오버레이 + 쿼리 텍스트와 함께 시각화"""
    scene_data = os.path.join(data_root, scene)
    if USE_AUTHOR_MODEL:
        scene_output = os.path.join(output_root, scene + "_author")
    else:
        scene_output = os.path.join(output_root, scene)
    r_dir = os.path.join(scene_output, "test_results", "renders")
    g_dir = os.path.join(scene_output, "test_results", "gt")
    img_dir = os.path.join(scene_data, "images")

    if not os.path.exists(r_dir):
        print("렌더링 결과 없음")
        return

    # test_json에서 카테고리별 문장+유형 로드
    json_dir = os.path.join(scene_data, "json", "test_json")
    if not os.path.isdir(json_dir):
        json_dir = os.path.join(scene_data, "test_json")

    cat_info = defaultdict(list)
    for jf in sorted(os.listdir(json_dir)):
        if not jf.endswith(".json"):
            continue
        with open(os.path.join(json_dir, jf)) as f:
            data = json.load(f)
        for obj in data.get("object", []):
            cat = obj.get("category", "")
            for sent in obj.get("sentence", []):
                stype = classify_sentence(sent)
                if not any(s["sentence"] == sent for s in cat_info[cat]):
                    cat_info[cat].append({"sentence": sent, "type": stype})

    # 결과 수집
    type_results = {"spatial": [], "attribute": [], "mixed": [], "simple": []}

    for fname in sorted(os.listdir(r_dir)):
        if not fname.endswith(".png"):
            continue
        gt_path = os.path.join(g_dir, fname)
        if not os.path.exists(gt_path):
            continue

        pred = np.array(Image.open(os.path.join(r_dir, fname)).convert("L")) > 0
        gt = np.array(Image.open(gt_path).convert("L")) > 0
        if gt.sum() == 0:
            continue

        intersection = np.logical_and(pred, gt).sum()
        union = np.logical_or(pred, gt).sum()
        iou = intersection / union * 100 if union > 0 else 0.0

        frame_num = int(''.join([c for c in fname.split('.')[0] if c.isdigit()]))
        cat = ''.join([c for c in fname.split('.png')[0] if not c.isdigit()]).strip()

        if cat in cat_info and cat_info[cat]:
            types = [s["type"] for s in cat_info[cat]]
            dominant_type = max(set(types), key=types.count)
            sentences = [s["sentence"] for s in cat_info[cat]]
        else:
            dominant_type = "simple"
            sentences = []

        type_results[dominant_type].append({
            "fname": fname, "frame": frame_num, "cat": cat,
            "iou": iou, "sentences": sentences,
            "pred_path": os.path.join(r_dir, fname),
            "gt_path": gt_path,
        })

    # 유형별 시각화
    for stype, results in type_results.items():
        if not results:
            continue

        results_sorted = sorted(results, key=lambda x: x["iou"])
        if len(results_sorted) >= n_per_type:
            selected = [results_sorted[0], results_sorted[len(results_sorted)//2], results_sorted[-1]]
            labels = ["WORST", "MID", "BEST"]
        else:
            selected = results_sorted
            labels = [f"#{i+1}" for i in range(len(selected))]

        n_show = len(selected)
        fig, axes = plt.subplots(n_show, 4, figsize=(20, 5 * n_show))
        if n_show == 1:
            axes = axes[np.newaxis, :]

        colors = {"spatial": "#e74c3c", "attribute": "#3498db", "mixed": "#9b59b6", "simple": "#2ecc71"}
        fig.suptitle(f'Type: {stype.upper()} ({len(results)} queries, avg IoU: {np.mean([r["iou"] for r in results]):.1f}%)',
                     fontsize=14, color=colors.get(stype, 'black'), fontweight='bold')

        for i, (r, label) in enumerate(zip(selected, labels)):
            rgb_path = os.path.join(img_dir, f"frame_{r['frame']:05d}.jpg")
            if not os.path.exists(rgb_path):
                rgb_path = os.path.join(img_dir, f"frame_{r['frame']:05d}.png")

            # 1) RGB
            if os.path.exists(rgb_path):
                rgb = np.array(Image.open(rgb_path))
                axes[i, 0].imshow(rgb)
            axes[i, 0].set_title(f"[{label}] RGB (frame {r['frame']})", fontsize=10)
            axes[i, 0].axis('off')

            # 2) RGB + 오버레이
            pred = np.array(Image.open(r["pred_path"]).convert("L")) > 0
            gt = np.array(Image.open(r["gt_path"]).convert("L")) > 0
            if os.path.exists(rgb_path):
                rgb_f = rgb.astype(float) / 255.0
                overlay = rgb_f * 0.5
                gt_3d = np.stack([np.zeros_like(gt), gt, np.zeros_like(gt)], axis=-1).astype(float)
                pred_3d = np.stack([pred, np.zeros_like(pred), np.zeros_like(pred)], axis=-1).astype(float)
                overlay = np.clip(overlay + gt_3d * 0.4 + pred_3d * 0.3, 0, 1)
                axes[i, 1].imshow(overlay)
                legend_patches = [mpatches.Patch(color='green', alpha=0.5, label='GT'),
                                  mpatches.Patch(color='red', alpha=0.5, label='Pred')]
                axes[i, 1].legend(handles=legend_patches, loc='upper right', fontsize=7)
            axes[i, 1].set_title(f"IoU: {r['iou']:.1f}% | {r['cat']}", fontsize=10)
            axes[i, 1].axis('off')

            # 3) Pred mask
            axes[i, 2].imshow(pred.astype(float), cmap='gray', vmin=0, vmax=1)
            axes[i, 2].set_title("Predicted", fontsize=10)
            axes[i, 2].axis('off')

            # 4) GT mask
            axes[i, 3].imshow(gt.astype(float), cmap='gray', vmin=0, vmax=1)
            axes[i, 3].set_title("Ground Truth", fontsize=10)
            axes[i, 3].axis('off')

        plt.tight_layout(rect=[0, 0.08, 1, 0.95])

        # 각 행 아래에 쿼리 텍스트 표시
        for i, r in enumerate(selected):
            if r["sentences"]:
                sent_text = r["sentences"][0]
                if len(sent_text) > 100:
                    sent_text = sent_text[:100] + "..."
            else:
                sent_text = "(no sentence)"
            y_pos = 1.0 - (i + 1) / n_show + 0.01
            fig.text(0.02, y_pos, f'Query: "{sent_text}"',
                     fontsize=8, color='blue', style='italic',
                     transform=fig.transFigure,
                     bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

        plt.show()

# 실행
for scene in ALL_SCENES:
    print(f"\n{'='*60}")
    print(f"Scene: {scene} - Expression Type Visual Results")
    print(f"{'='*60}")
    visualize_expression_types(DATA_ROOT, OUTPUT_ROOT, scene)

### 11-4. 뷰 일관성 분석
동일 객체가 여러 테스트 뷰에서 얼마나 일관된 성능을 보이는지 분석합니다.
뷰에 따라 IoU 편차가 크다면, 3D scene knowledge 활용이 불충분하다는 증거입니다.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from collections import defaultdict

# ============================================================
# 실험 5: 뷰 일관성 (ALL_RESULTS 재사용)
# ============================================================
if ALL_RESULTS:
    obj_views = defaultdict(list)
    for r in ALL_RESULTS:
        obj_views[r["category"]].append(r)

    # 2개 이상 뷰에 등장하는 객체만
    multi = {k: v for k, v in obj_views.items() if len(v) >= 2}
    obj_names = sorted(multi.keys())

    obj_means = [np.mean([r["iou"] for r in multi[o]]) for o in obj_names]
    obj_stds = [np.std([r["iou"] for r in multi[o]]) for o in obj_names]
    obj_ranges = [max(r["iou"] for r in multi[o]) - min(r["iou"] for r in multi[o]) for o in obj_names]
    obj_cvs = [s/m*100 if m > 0 else 0 for s, m in zip(obj_stds, obj_means)]

    # 시각화
    fig, ax = plt.subplots(figsize=(14, 6))
    for i, obj in enumerate(obj_names):
        ious = [r["iou"] for r in multi[obj]]
        ax.scatter([i]*len(ious), ious, alpha=0.6, s=40, zorder=3)
        ax.plot([i, i], [min(ious), max(ious)], 'k-', alpha=0.3, linewidth=2)
        ax.plot(i, np.mean(ious), 'D', color='red', markersize=8, zorder=4)
    ax.set_xticks(range(len(obj_names)))
    ax.set_xticklabels(obj_names, rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('IoU (%)', fontsize=12)
    ax.set_title(f'{SCENE} - Per-Object IoU Across Views (red diamond = mean)', fontsize=13)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

    # 테이블 (변동계수 CV 추가)
    print(f"\n{'객체':<20} {'뷰수':<5} {'mIoU':<8} {'std':<8} {'범위':<8} {'CV':<8} {'안정성'}")
    print("-" * 65)
    for obj, mean, std, rng, cv in zip(obj_names, obj_means, obj_stds, obj_ranges, obj_cvs):
        stability = "안정" if cv < 30 else ("불안정" if cv < 80 else "매우불안정")
        n = len(multi[obj])
        print(f"  {obj:<18} {n:<5} {mean:<8.1f} {std:<8.1f} {rng:<8.1f} {cv:<8.1f} {stability}")

    print(f"\n평균 CV: {np.mean(obj_cvs):.1f}%  |  평균 범위: {np.mean(obj_ranges):.1f}%")
else:
    print("ALL_RESULTS 없음.")

### 11-5. 실패 케이스 시각화
IoU가 가장 낮은 케이스와 가장 높은 케이스를 나란히 보여줍니다.
모델이 어떤 상황에서 실패하고 성공하는지 시각적으로 비교합니다.

In [ ]:
import os, torch, numpy as np, matplotlib.pyplot as plt
from PIL import Image

def collect_all_results(output_root, scenes):
    """전체 결과를 (scene, filename, iou, pred_path, gt_path)로 수집"""
    all_results = []
    for scene in scenes:
        scene_dir = os.path.join(output_root, scene + "_author") if USE_AUTHOR_MODEL else os.path.join(output_root, scene)
        r_dir = os.path.join(scene_dir, "test_results", "renders")
        g_dir = os.path.join(scene_dir, "test_results", "gt")
        if not os.path.exists(r_dir):
            continue
        for fname in sorted(os.listdir(r_dir)):
            if not fname.endswith(".png"):
                continue
            gt_path = os.path.join(g_dir, fname)
            if not os.path.exists(gt_path):
                continue
            pred = torch.from_numpy(np.array(Image.open(os.path.join(r_dir, fname)).convert("L"))).float() > 0
            gt = torch.from_numpy(np.array(Image.open(gt_path).convert("L"))).float() > 0
            if gt.sum() == 0:
                continue
            intersection = torch.logical_and(pred, gt).sum().float()
            union = torch.logical_or(pred, gt).sum().float()
            iou = (intersection / union).item() * 100 if union > 0 else 0.0
            all_results.append({
                "scene": scene, "fname": fname, "iou": iou,
                "pred_path": os.path.join(r_dir, fname),
                "gt_path": gt_path
            })
    return sorted(all_results, key=lambda x: x["iou"])

results = collect_all_results(OUTPUT_ROOT, ALL_SCENES)
if not results:
    print("결과가 없습니다.")
else:
    n_show = 5

    # 최악 케이스
    worst = results[:n_show]
    # 최고 케이스
    best = results[-n_show:][::-1]

    fig, axes = plt.subplots(n_show, 4, figsize=(16, 4 * n_show))
    fig.suptitle('Worst Cases (left) vs Best Cases (right)', fontsize=16, y=1.01)

    for i in range(n_show):
        # Worst - Pred
        w_pred = np.array(Image.open(worst[i]["pred_path"]).convert("L"))
        w_gt = np.array(Image.open(worst[i]["gt_path"]).convert("L"))
        axes[i, 0].imshow(w_pred, cmap='gray')
        axes[i, 0].set_title(f"WORST Pred: {worst[i]['scene']}/{worst[i]['fname']}\nIoU: {worst[i]['iou']:.1f}%", fontsize=8)
        axes[i, 0].axis('off')
        axes[i, 1].imshow(w_gt, cmap='gray')
        axes[i, 1].set_title(f"GT", fontsize=8)
        axes[i, 1].axis('off')

        # Best - Pred
        b_pred = np.array(Image.open(best[i]["pred_path"]).convert("L"))
        b_gt = np.array(Image.open(best[i]["gt_path"]).convert("L"))
        axes[i, 2].imshow(b_pred, cmap='gray')
        axes[i, 2].set_title(f"BEST Pred: {best[i]['scene']}/{best[i]['fname']}\nIoU: {best[i]['iou']:.1f}%", fontsize=8)
        axes[i, 2].axis('off')
        axes[i, 3].imshow(b_gt, cmap='gray')
        axes[i, 3].set_title(f"GT", fontsize=8)
        axes[i, 3].axis('off')

    plt.tight_layout()
    plt.show()

    # IoU 분포 히스토그램
    all_ious = [r["iou"] for r in results]
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(all_ious, bins=20, color='steelblue', edgecolor='black', alpha=0.7)
    ax.axvline(np.mean(all_ious), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(all_ious):.1f}%')
    ax.axvline(np.median(all_ious), color='orange', linestyle='--', linewidth=2, label=f'Median: {np.median(all_ious):.1f}%')
    ax.set_xlabel('IoU (%)', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.set_title('IoU Distribution Across All Scenes', fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

    # 통계 요약
    print(f"\n전체 쿼리 수: {len(all_ious)}")
    print(f"Mean IoU: {np.mean(all_ious):.1f}%")
    print(f"Median IoU: {np.median(all_ious):.1f}%")
    print(f"IoU = 0% 비율: {sum(1 for x in all_ious if x < 1) / len(all_ious) * 100:.1f}%")
    print(f"IoU > 50% 비율: {sum(1 for x in all_ious if x > 50) / len(all_ious) * 100:.1f}%")

### 11-6. 자유 텍스트 쿼리 실험
학습에 사용되지 않은 임의의 텍스트를 쿼리로 넣어서 모델이 어떻게 반응하는지 확인합니다.
모델의 일반화 능력(generalization)을 테스트하는 실험입니다.

In [ ]:
%%writefile {REPO_DIR}/free_query.py
"""
자유 텍스트 쿼리로 세그멘테이션 실행
Usage: python free_query.py -s <data_path> -m <output_path> --checkpoint_file <ckpt> --queries "food" "bowl" "red object"
"""
import re, os, sys, json, random
import numpy as np
import torch
import torch.nn.functional as F
import torchvision
from scene import Scene
from gaussian_renderer import render, GaussianModel
from utils.general_utils import safe_state
from argparse import ArgumentParser
from arguments import ModelParams, PipelineParams, OptimizationParams
from tqdm import tqdm
from os import makedirs

if __name__ == "__main__":
    random.seed(0)
    np.random.seed(0)
    torch.manual_seed(0)
    torch.cuda.manual_seed_all(0)

    parser = ArgumentParser()
    lp = ModelParams(parser)
    op = OptimizationParams(parser)
    pp = PipelineParams(parser)
    parser.add_argument("--checkpoint_file", type=str, required=True)
    parser.add_argument("--queries", nargs="+", type=str, required=True)
    parser.add_argument("--test_frame_indices", nargs="+", type=int, default=[6, 60, 128])
    parser.add_argument("--quiet", action="store_true")
    args = parser.parse_args()
    args.include_feature = True

    safe_state(args.quiet)

    with torch.no_grad():
        dataset = lp.extract(args)
        gaussians = GaussianModel(dataset.sh_degree)
        scene = Scene(dataset, gaussians, shuffle=False)

        (model_params, first_iter) = torch.load(
            args.checkpoint_file,
            map_location=f'cuda:{torch.cuda.current_device()}',
            weights_only=False
        )
        gaussians.restore(model_params, args, mode='test')

        bg_color = [1, 1, 1] if dataset.white_background else [0, 0, 0]
        background = torch.tensor(bg_color, dtype=torch.float32, device="cuda")

        # 테스트 카메라에서 지정된 프레임만 선택
        test_cams = scene.getTestCameras()

        # 출력 디렉토리
        out_dir = os.path.join(dataset.model_path, "free_query_results")
        makedirs(out_dir, exist_ok=True)

        for cam in tqdm(test_cams, desc="Processing views"):
            frame_num = int(re.findall(r'\d+', cam.image_name)[0])
            if frame_num not in args.test_frame_indices:
                continue

            for query in args.queries:
                output = render(cam, gaussians, pp.extract(args), background, args, sentence=query)
                seg_map = output["language_feature_image"]
                seg_map = torch.sigmoid(seg_map)
                seg_map = (seg_map >= 0.5).float()

                # 쿼리에서 파일명 안전하게 만들기
                safe_query = query.replace(" ", "_").replace("/", "_")
                fname = f"frame{frame_num:05d}_{safe_query}.png"
                torchvision.utils.save_image(seg_map, os.path.join(out_dir, fname))

        print(f"Free query results saved to: {out_dir}")

In [ ]:
%cd {REPO_DIR}
import glob

# 13개 카테고리 단어 + 추가 쿼리
CATEGORY_QUERIES = [
    "bowl", "chopsticks", "corn", "egg", "eggs",
    "glass of water", "hand", "kamaboko", "nori",
    "plate", "sake cup", "spoon", "wavy noodles",
]
EXTRA_QUERIES = [
    "food",                    # 상위 카테고리
    "yellow object",           # 색상 기반
    "wooden object",           # 재질 기반
    "object on the left",      # 순수 공간 표현
    "table",                   # 배경 객체
    "car",                     # 장면에 없는 객체
]
FREE_QUERIES = CATEGORY_QUERIES + EXTRA_QUERIES

# 테스트할 뷰 (7개 전부)
TEST_FRAMES = [6, 24, 60, 65, 81, 119, 128]

# 체크포인트 결정
if USE_AUTHOR_MODEL:
    latest_ckpt = os.path.join(CHECKPOINT_ROOT, AUTHOR_CKPT_MAP[SCENE])
    ckpt_exists = os.path.exists(latest_ckpt)
else:
    ckpt_files = sorted(glob.glob(os.path.join(EVAL_OUTPUT_PATH, "chkpnt*.pth")))
    latest_ckpt = ckpt_files[-1] if ckpt_files else None
    ckpt_exists = latest_ckpt is not None

if ckpt_exists:
    queries_str = " ".join([f'"{q}"' for q in FREE_QUERIES])
    frames_str = " ".join([str(f) for f in TEST_FRAMES])
    print(f"Using checkpoint: {latest_ckpt}")
    print(f"Queries: {len(FREE_QUERIES)}개 ({len(CATEGORY_QUERIES)} categories + {len(EXTRA_QUERIES)} extra)")
    print(f"Test frames: {TEST_FRAMES}")
    !python free_query.py \
        -s {SCENE_DATA_PATH} \
        -m {EVAL_OUTPUT_PATH} \
        --checkpoint_file {latest_ckpt} \
        --queries {queries_str} \
        --test_frame_indices {frames_str}
else:
    print("체크포인트가 없습니다.")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import numpy as np
import os, re

def visualize_free_queries(data_path, output_path, queries, test_frames):
    """자유 쿼리 결과: RGB 오버레이 + 흑백 마스크 두 행으로 시각화"""
    result_dir = os.path.join(output_path, "free_query_results")
    img_dir = os.path.join(data_path, "images")

    if not os.path.exists(result_dir):
        print("자유 쿼리 결과가 없습니다. 위 셀을 먼저 실행하세요.")
        return

    n_queries = len(queries)

    for frame in test_frames:
        # RGB 이미지 경로
        rgb_path = os.path.join(img_dir, f"frame_{frame:05d}.jpg")
        if not os.path.exists(rgb_path):
            rgb_path = os.path.join(img_dir, f"frame_{frame:05d}.png")

        # 2행: 위=오버레이, 아래=흑백 마스크
        fig, axes = plt.subplots(2, n_queries + 1, figsize=(3.5 * (n_queries + 1), 7))
        fig.suptitle(f"Frame {frame} - Free Text Query Results", fontsize=14)

        # 첫 열: RGB 원본 (위아래 동일)
        if os.path.exists(rgb_path):
            rgb = np.array(Image.open(rgb_path))
            axes[0, 0].imshow(rgb)
            axes[1, 0].imshow(rgb)
        axes[0, 0].set_title("RGB Original", fontsize=10)
        axes[1, 0].set_title("RGB Original", fontsize=10)
        axes[0, 0].axis('off')
        axes[1, 0].axis('off')

        for j, query in enumerate(queries):
            safe_query = query.replace(" ", "_").replace("/", "_")
            mask_path = os.path.join(result_dir, f"frame{frame:05d}_{safe_query}.png")

            if os.path.exists(mask_path):
                mask = np.array(Image.open(mask_path).convert("L")) > 0
                mask_ratio = mask.sum() / mask.size * 100

                # 위 행: RGB + 빨간 오버레이
                if os.path.exists(rgb_path):
                    rgb_img = np.array(Image.open(rgb_path)).astype(float) / 255.0
                    overlay = rgb_img.copy() * 0.5
                    mask_3d = np.stack([mask, np.zeros_like(mask), np.zeros_like(mask)], axis=-1).astype(float)
                    overlay = np.clip(overlay + mask_3d * 0.5, 0, 1)
                    axes[0, j + 1].imshow(overlay)
                axes[0, j + 1].set_title(f'"{query}"\n({mask_ratio:.1f}%)', fontsize=9)
                axes[0, j + 1].axis('off')

                # 아래 행: 흑백 마스크
                axes[1, j + 1].imshow(mask.astype(float), cmap='gray', vmin=0, vmax=1)
                axes[1, j + 1].set_title(f"Mask", fontsize=9)
                axes[1, j + 1].axis('off')
            else:
                axes[0, j + 1].text(0.5, 0.5, "No result", ha='center', va='center',
                                     transform=axes[0, j + 1].transAxes)
                axes[0, j + 1].set_title(f'"{query}"', fontsize=9)
                axes[0, j + 1].axis('off')
                axes[1, j + 1].axis('off')

        plt.tight_layout()
        plt.show()

# 실행
visualize_free_queries(SCENE_DATA_PATH, EVAL_OUTPUT_PATH, FREE_QUERIES, TEST_FRAMES)

#### 짧은 단어 쿼리 정량 평가
카테고리 이름만으로 쿼리했을 때 GT 마스크와의 IoU를 계산합니다.
학습 문장에서 해당 단어 포함률과 IoU의 상관관계를 분석합니다.

In [ ]:
import os, torch, numpy as np, matplotlib.pyplot as plt
from PIL import Image

# ============================================================
# 실험 6: 짧은 단어 쿼리 정량 평가 (ALL_RESULTS 재사용)
# ============================================================
# 학습 문장에서 카테고리 단어 포함률 (개별 단어 하나라도 포함 기준)
WORD_INCLUSION_RATE = {
    "bowl": 25, "chopsticks": 75, "corn": 25, "egg": 100, "eggs": 0,
    "glass of water": 100, "hand": 100, "kamaboko": 0, "nori": 0,
    "plate": 25, "sake cup": 25, "spoon": 50, "wavy noodles": 100,
}
# 참고: glass of water → "glass","water" 개별 포함 100%
#        wavy noodles → "noodles" 포함 100%
#        eggs → "eggs" 단어 자체는 0% (문장에서 "egg"만 사용)

def evaluate_short_query_iou(data_path, output_path, categories, test_frames):
    """카테고리 단어 쿼리의 IoU를 GT 마스크와 비교"""
    result_dir = os.path.join(output_path, "free_query_results")
    gt_mask_dir = os.path.join(data_path, "gt_mask")
    if not os.path.exists(result_dir):
        print("자유 쿼리 결과 없음. 위 셀을 먼저 실행하세요.")
        return None
    results = []
    for cat in categories:
        cat_ious = []
        safe_query = cat.replace(" ", "_").replace("/", "_")
        for frame in test_frames:
            pred_path = os.path.join(result_dir, f"frame{frame:05d}_{safe_query}.png")
            gt_path = os.path.join(gt_mask_dir, f"frame_{frame:05d}", f"{cat}.png")
            if not os.path.exists(pred_path) or not os.path.exists(gt_path):
                continue
            pred = torch.from_numpy(np.array(Image.open(pred_path).convert("L"))).float() > 0
            gt = torch.from_numpy(np.array(Image.open(gt_path).convert("L"))).float() > 0
            if gt.sum() == 0:
                continue
            intersection = torch.logical_and(pred, gt).sum().float().item()
            union = torch.logical_or(pred, gt).sum().float().item()
            iou = intersection / union * 100 if union > 0 else 0.0
            cat_ious.append(iou)
        if cat_ious:
            results.append({
                "category": cat, "mean_iou": np.mean(cat_ious), "std_iou": np.std(cat_ious),
                "n_views": len(cat_ious), "word_rate": WORD_INCLUSION_RATE.get(cat, -1),
            })
    return results

short_results = evaluate_short_query_iou(
    os.path.join(DATA_ROOT, SCENE), EVAL_OUTPUT_PATH,
    CATEGORY_QUERIES, TEST_FRAMES
)

if short_results and ALL_RESULTS:
    # === 긴 문장 IoU vs 짧은 단어 IoU 비교 ===
    print(f"\n{'=' * 80}")
    print(f"실험 6: 긴 문장 쿼리 vs 짧은 단어 쿼리 비교 (Scene: {SCENE})")
    print(f"{'=' * 80}")
    print(f"{'카테고리':<20} {'단어포함률':>8} {'긴문장IoU':>10} {'짧은단어IoU':>10} {'Gap':>8}")
    print("-" * 60)

    from collections import defaultdict
    long_by_cat = defaultdict(list)
    for r in ALL_RESULTS:
        long_by_cat[r["category"]].append(r["iou"])

    gaps = []
    rates = []
    short_ious_for_corr = []
    for sr in sorted(short_results, key=lambda x: x["mean_iou"], reverse=True):
        cat = sr["category"]
        long_miou = np.mean(long_by_cat[cat]) if cat in long_by_cat else 0
        gap = long_miou - sr["mean_iou"]
        gaps.append(gap)
        rates.append(sr["word_rate"])
        short_ious_for_corr.append(sr["mean_iou"])
        print(f"  {cat:<18} {sr['word_rate']:>7}% {long_miou:>9.1f}% {sr['mean_iou']:>9.1f}% {gap:>+7.1f}%")

    long_overall = np.mean([r["iou"] for r in ALL_RESULTS])
    short_overall = np.mean([r["mean_iou"] for r in short_results])
    print(f"\n  {'전체':<18} {'':>8} {long_overall:>9.1f}% {short_overall:>9.1f}% {long_overall-short_overall:>+7.1f}%")

    # 상관관계
    corr = np.corrcoef(rates, short_ious_for_corr)[0, 1]
    print(f"\n  단어포함률 vs 짧은단어IoU 상관계수: {corr:.3f}")

    # Scatter plot
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # 좌: 단어 포함률 vs 짧은 단어 IoU
    axes[0].scatter(rates, short_ious_for_corr, c=short_ious_for_corr, cmap='RdYlGn', s=100, edgecolors='black', zorder=3)
    for sr in short_results:
        axes[0].annotate(sr["category"], (sr["word_rate"], sr["mean_iou"]),
                        fontsize=8, ha='center', va='bottom',
                        bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))
    z = np.polyfit(rates, short_ious_for_corr, 1)
    p = np.poly1d(z)
    axes[0].plot(np.linspace(0, 100, 100), p(np.linspace(0, 100, 100)), '--', color='gray', alpha=0.5)
    axes[0].set_xlabel('학습 문장 내 단어 포함률 (%)', fontsize=11)
    axes[0].set_ylabel('짧은 단어 쿼리 IoU (%)', fontsize=11)
    axes[0].set_title(f'Word Inclusion Rate vs Short Query IoU (r={corr:.2f})', fontsize=12)
    axes[0].grid(True, alpha=0.3)
    axes[0].set_xlim(-5, 105)

    # 우: 긴 문장 IoU vs 짧은 단어 IoU (객체별)
    long_means = []
    short_means = []
    cat_labels = []
    for sr in short_results:
        cat = sr["category"]
        if cat in long_by_cat:
            long_means.append(np.mean(long_by_cat[cat]))
            short_means.append(sr["mean_iou"])
            cat_labels.append(cat)
    axes[1].scatter(long_means, short_means, c='steelblue', s=100, edgecolors='black', zorder=3)
    for lm, sm, cl in zip(long_means, short_means, cat_labels):
        axes[1].annotate(cl, (lm, sm), fontsize=8, ha='center', va='bottom',
                        bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))
    axes[1].plot([0, 100], [0, 100], '--', color='gray', alpha=0.5, label='y=x (no gap)')
    axes[1].set_xlabel('긴 문장 쿼리 IoU (%)', fontsize=11)
    axes[1].set_ylabel('짧은 단어 쿼리 IoU (%)', fontsize=11)
    axes[1].set_title('Long Sentence vs Short Word Query IoU', fontsize=12)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("결과 없음.")

### 11-7. 종합 분석 요약
위 실험들의 결과를 하나의 테이블로 정리합니다.

In [ ]:
print("=" * 70)
print("ReferSplat 문제점 분석 종합 요약")
print("=" * 70)

print("""
[문제 1] 객체 크기 의존성
  - 큰 객체(>15% 면적): 높은 IoU
  - 작은 객체(<1% 면적): 거의 0% IoU
  → 16차원 referring feature가 작은 객체의 세밀한 특징을 표현하기 어려움
  → Gaussian 포인트 밀도가 작은 객체에서 부족

[문제 2] 객체 유형별 한계
  - 투명/반사 객체: 3DGS 자체가 복원 못하므로 세그멘테이션 불가
  - 가늘고 긴 객체: Gaussian 표현의 근본적 한계
  → 3DGS 기반 방법의 구조적 한계

[문제 3] 공간 관계 이해 부족
  - Spatial 표현 포함 문장의 성능이 낮을 경우:
    Position-aware Cross-Modal Interaction이 실제 공간 관계를
    충분히 모델링하지 못하고 있음을 시사
  → 논문의 핵심 기여에 대한 의문 제기 가능

[문제 4] 뷰 일관성 부족
  - 동일 객체의 뷰별 IoU 편차가 큰 경우:
    3D scene knowledge 활용이 불충분
  - 특정 뷰에서만 잘 되고 다른 뷰에서 실패
  → multi-view consistency 강화 필요

[문제 5] 성능 양극화
  - IoU 분포가 bimodal (0% 근처 + 80% 이상)
  - "되는 건 잘 되고, 안 되는 건 아예 안 됨"
  → pseudo mask 품질이 성능 상한선을 결정하는 병목
""")

print("=" * 70)